# `test_clustering.py` — Unit Tests for `EmployeeClusterer`

## Purpose

Validates the K-Means clustering pipeline that identifies departure archetypes among employees
who left. Tests cover data preparation (leaver filtering), model fitting, cluster summary
aggregation, archetype interpretation, and the end-to-end pipeline method.

---

## Module Under Test

`src.clustering.employee_clusterer.EmployeeClusterer`

---

## Test Classes at a Glance

| Class | Methods Tested | What It Verifies |
|-------|----------------|-----------------|
| `TestPrepareClusterData` | `prepare_cluster_data()` | Only leavers included, 2-column output, no-leavers error |
| `TestFitKmeans` | `fit_kmeans()` | Label length, cluster count, pre-condition guard, model storage |
| `TestGetClusterSummary` | `get_cluster_summary()` | Shape, required columns, count integrity |
| `TestInterpretClusters` | `interpret_clusters()` | Dict keys, label + description per cluster |
| `TestRunClusteringPipeline` | `run_clustering_pipeline()` | Returns dict with summary, interpretation, model |

---

## Fixtures Used

| Fixture | Source |
|---------|--------|
| `sample_df` | `conftest.py` |

---

## How to Run

```bash
pytest tests/test_clustering.py -v
```


---

## `TestPrepareClusterData`

**Purpose:** Tests `prepare_cluster_data()` — verifies that the method correctly filters the
DataFrame to only include employees who left (`left=1`) and returns exactly the two features
used for clustering (`satisfaction_level`, `last_evaluation`).

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_only_leavers_included` | Row count equals number of `left=1` employees in `sample_df` |
| `test_only_two_feature_columns` | Columns are exactly `{satisfaction_level, last_evaluation}` |
| `test_no_leavers_raises` | Raises `ClusteringError` when all employees have `left=0` |
| `test_returns_dataframe` | Return type is `pd.DataFrame` |


In [ ]:
import pandas as pd
import pytest

from src.clustering.employee_clusterer import EmployeeClusterer
from src.utils.exceptions import ClusteringError


class TestPrepareClusterData:
    def test_only_leavers_included(self, sample_df):
        clusterer = EmployeeClusterer(sample_df)
        data = clusterer.prepare_cluster_data()
        leavers_count = (sample_df["left"] == 1).sum()
        assert len(data) == leavers_count

    def test_only_two_feature_columns(self, sample_df):
        clusterer = EmployeeClusterer(sample_df)
        data = clusterer.prepare_cluster_data()
        assert set(data.columns) == {"satisfaction_level", "last_evaluation"}

    def test_no_leavers_raises(self, sample_df):
        df_no_leavers = sample_df.copy()
        df_no_leavers["left"] = 0
        clusterer = EmployeeClusterer(df_no_leavers)
        with pytest.raises(ClusteringError):
            clusterer.prepare_cluster_data()

    def test_returns_dataframe(self, sample_df):
        clusterer = EmployeeClusterer(sample_df)
        result = clusterer.prepare_cluster_data()
        assert isinstance(result, pd.DataFrame)


---

## `TestFitKmeans`

**Purpose:** Tests `fit_kmeans()` — verifies that K-Means produces the correct number of
cluster labels, that it requires `prepare_cluster_data()` to be called first, and that the
fitted model is stored on the instance.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_labels_length_matches_data` | `len(clusterer.labels)` equals `len(clusterer.cluster_data)` |
| `test_correct_number_of_clusters` | Exactly `n_clusters` distinct labels are produced |
| `test_fit_without_prepare_raises` | Raises `ClusteringError` with "prepare_cluster_data" message |
| `test_model_stored_on_instance` | `clusterer.kmeans_model` is the same object as the return value |


In [ ]:
class TestFitKmeans:
    def test_labels_length_matches_data(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        assert len(clusterer.labels) == len(clusterer.cluster_data)

    def test_correct_number_of_clusters(self, sample_df):
        n = 3
        clusterer = EmployeeClusterer(sample_df, n_clusters=n)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        assert len(set(clusterer.labels)) == n

    def test_fit_without_prepare_raises(self, sample_df):
        clusterer = EmployeeClusterer(sample_df)
        with pytest.raises(ClusteringError, match="prepare_cluster_data"):
            clusterer.fit_kmeans()

    def test_model_stored_on_instance(self, sample_df):
        clusterer = EmployeeClusterer(sample_df)
        clusterer.prepare_cluster_data()
        model = clusterer.fit_kmeans()
        assert clusterer.kmeans_model is model


---

## `TestGetClusterSummary` · `TestInterpretClusters`

**Purpose:**
- `TestGetClusterSummary` verifies the aggregated summary DataFrame has one row per cluster,
  contains the expected columns, and that employee counts across clusters sum to total leavers.
- `TestInterpretClusters` verifies that each cluster receives a human-readable `label` and
  `description` describing the departure archetype.

### Test Methods

| Method | Class | Verifies |
|--------|-------|----------|
| `test_summary_shape` | `TestGetClusterSummary` | `len(summary) == 3` |
| `test_summary_columns` | `TestGetClusterSummary` | Contains `satisfaction_level`, `last_evaluation`, `count` |
| `test_counts_sum_to_total_leavers` | `TestGetClusterSummary` | `count.sum()` equals total `left=1` employees |
| `test_returns_dict_with_cluster_keys` | `TestInterpretClusters` | Returns dict with 3 keys |
| `test_each_cluster_has_label` | `TestInterpretClusters` | Each value has `label` and `description` |


In [ ]:
class TestGetClusterSummary:
    def test_summary_shape(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        summary = clusterer.get_cluster_summary()
        assert len(summary) == 3

    def test_summary_columns(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        summary = clusterer.get_cluster_summary()
        assert "satisfaction_level" in summary.columns
        assert "last_evaluation" in summary.columns
        assert "count" in summary.columns

    def test_counts_sum_to_total_leavers(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        summary = clusterer.get_cluster_summary()
        total_leavers = (sample_df["left"] == 1).sum()
        assert summary["count"].sum() == total_leavers


class TestInterpretClusters:
    def test_returns_dict_with_cluster_keys(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        interpretations = clusterer.interpret_clusters()
        assert len(interpretations) == 3

    def test_each_cluster_has_label(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        clusterer.prepare_cluster_data()
        clusterer.fit_kmeans()
        interpretations = clusterer.interpret_clusters()
        for cluster_id, info in interpretations.items():
            assert "label" in info
            assert "description" in info


---

## `TestRunClusteringPipeline`

**Purpose:** Tests the convenience orchestration method `run_clustering_pipeline()` — verifies
it chains all steps internally and returns a single dictionary with all three required keys.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_pipeline_returns_dict` | Returns `dict` with keys `summary`, `interpretation`, `model` |


In [ ]:
class TestRunClusteringPipeline:
    def test_pipeline_returns_dict(self, sample_df):
        clusterer = EmployeeClusterer(sample_df, n_clusters=3)
        result = clusterer.run_clustering_pipeline()
        assert isinstance(result, dict)
        assert "summary" in result
        assert "interpretation" in result
        assert "model" in result
